In [3]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import joblib
from xgboost import XGBRegressor

# --- NEW: MLFLOW SILENT OBSERVERS ---
import mlflow
import mlflow.sklearn
import mlflow.xgboost

# Point to your Docker Tracking Server
mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("SmartCalorie_AI_Training_v2")

# Enable Autologging (The "Security Cameras")
mlflow.sklearn.autolog(log_models=True)
mlflow.xgboost.autolog(log_models=True)
# ------------------------------------

# 1. Load train / test data (UNTOUCHED)
BASE_DIR = Path("..").resolve()
DATA_DIR = BASE_DIR / "data" / "gold"
ARTIFACTS_DIR = BASE_DIR / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

X_train = pd.read_csv(DATA_DIR / "X_train.csv")
X_test = pd.read_csv(DATA_DIR / "X_test.csv")
y_train = pd.read_csv(DATA_DIR / "y_train.csv").values.ravel()
y_test = pd.read_csv(DATA_DIR / "y_test.csv").values.ravel()

with mlflow.start_run(run_name="Model_Comparison_XGB_vs_RF") as run:
    
    # 2. Define models (UNTOUCHED)
    rf = RandomForestRegressor(random_state=42, n_jobs=-1)
    rf_param_grid = {"n_estimators": [100, 200], "max_depth": [None, 10, 20], "min_samples_split": [2, 5]}

    xgb = XGBRegressor(objective="reg:squarederror", random_state=42, n_jobs=-1)
    xgb_param_grid = {"n_estimators": [100, 200], "max_depth": [3, 6], "learning_rate": [0.05, 0.1]}

    # 3. Run GridSearchCV (UNTOUCHED - MLflow logs this automatically)
    cv_folds = 3
    rf_grid = GridSearchCV(estimator=rf, param_grid=rf_param_grid, cv=cv_folds, scoring="r2", n_jobs=-1, verbose=1)
    xgb_grid = GridSearchCV(estimator=xgb, param_grid=xgb_param_grid, cv=cv_folds, scoring="r2", n_jobs=-1, verbose=1)

    print("\nFitting Models...")
    rf_grid.fit(X_train, y_train)
    xgb_grid.fit(X_train, y_train)

    # 4. Evaluate (UNTOUCHED)
    rf_best = rf_grid.best_estimator_
    xgb_best = xgb_grid.best_estimator_
    rf_preds = rf_best.predict(X_test)
    xgb_preds = xgb_best.predict(X_test)

    def evaluate_model(name: str, y_true, y_pred):
        r2 = r2_score(y_true, y_pred)
        print(f"\n{name} R^2: {r2:.4f}")
        return r2

    rf_r2 = evaluate_model("RandomForestRegressor", y_test, rf_preds)
    xgb_r2 = evaluate_model("XGBRegressor", y_test, xgb_preds)

    # 5. Pick the winner and save (UNTOUCHED)
    if xgb_r2 > rf_r2:
        best_model, best_name, best_r2 = xgb_best, "XGBRegressor", xgb_r2
    else:
        best_model, best_name, best_r2 = rf_best, "RandomForestRegressor", rf_r2

    # Save local fallback
    joblib.dump(best_model, ARTIFACTS_DIR / "best_model_1.pkl")
    
    # --- NEW: LOG THE WINNER TO MLFLOW ---
    mlflow.set_tag("winner", best_name)
    mlflow.log_metric("final_test_r2", best_r2)
    
    # Register the winning model in the MLflow Registry
    model_uri = f"runs:/{run.info.run_id}/model"
    mlflow.register_model(model_uri, "CaloriePredictor")
    
    print(f"\nWinner: {best_name} registered in MLflow.")

2026/03/18 06:13:26 INFO mlflow.tracking.fluent: Experiment with name 'SmartCalorie_AI_Training_v2' does not exist. Creating a new experiment.
2026/03/18 06:13:27 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\kaout\AppData\Local\Programs\Python\Python313\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for 


Fitting Models...
Fitting 3 folds for each of 12 candidates, totalling 36 fits


2026/03/18 06:13:34 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\kaout\AppData\Local\Programs\Python\Python313\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2026/03/18 06:13:41 INFO mlflow.sklearn.utils: Logging the 5 best runs, 7 runs will be omitted.
2026/03/18 06:13:41 WARNING mlfl

Fitting 3 folds for each of 8 candidates, totalling 24 fits


2026/03/18 06:13:44 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\kaout\AppData\Local\Programs\Python\Python313\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2026/03/18 06:13:52 INFO mlflow.sklearn.utils: Logging the 5 best runs, 3 runs will be omitted.
2026/03/18 06:13:52 WARNING mlfl


RandomForestRegressor R^2: 0.9853

XGBRegressor R^2: 0.9866


Registered model 'CaloriePredictor' already exists. Creating a new version of this model...
2026/03/18 06:13:53 WARNING mlflow.tracking._model_registry.fluent: Run with id a69c8913c41b455982b081dec8fe75c4 has no artifacts at artifact path 'model', registering model based on models:/m-1fd65b4975fc4cae9bb5d3757df41eb7 instead
2026/03/18 06:13:53 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: CaloriePredictor, version 1



Winner: XGBRegressor registered in MLflow.
🏃 View run Model_Comparison_XGB_vs_RF at: http://localhost:5000/#/experiments/2/runs/a69c8913c41b455982b081dec8fe75c4
🧪 View experiment at: http://localhost:5000/#/experiments/2


Created version '1' of model 'CaloriePredictor'.
